## 🎉 Congratulations!

You've made it this far—well done! Before continuing, make sure you have completed Notebook 05.

## Learning Objectives

By the end of this notebook, you will be able to:

- Build an NVIDIA NIM container image using Apptainer.
- Deploy a local NIM inference endpoint as a Slurm job on a GPU node.
- Configure the NIM API key, cache directory, port, and required container bindings.
- Connect to a local NIM endpoint using LangChain's `ChatNVIDIA`.
- Send inference requests directly through the OpenAI-compatible REST API.
- Parse and validate responses returned by the local inference endpoint.

### Building the NIM Apptainer Image

Athena does not support Docker directly. Therefore, the NVIDIA NIM Docker image must first be converted into an Apptainer `.sif` image before it can run on the cluster.

This conversion has already been completed for this tutorial, so **you do not need to run the script below**. It is included only as a reference example showing how the NIM image was pulled from NVIDIA's container registry and converted to Apptainer's SIF format.

### Deploying the NIM Endpoint as a Slurm Job

The cell below writes `01_deploy_nim.sh` — a Slurm batch script that launches the NIM Apptainer container on a GPU node. Once submitted, NIM will listen on port `8000` inside the container.

**Before submitting, update two placeholders in the script:**

- `PATH_TO_IMAGE_DIRECTORY` — where the pre-built `.sif` image lives.
- `PATH_TO_CACHE_DIRECTORY` — a writable per-user path on scratch for the NIM model cache.

**Then export your NGC key and submit:**

```bash
export NGC_API_KEY=nvapi-...
sbatch 01_deploy_nim.sh
```

**Key things the script does:**

- Requests 1 GPU, 8 CPUs, 64 GB RAM for up to 1 hour on the `tutorial` partition.
- Forwards `NGC_API_KEY`, `NIM_CACHE_PATH`, and `NIM_HTTP_API_PORT` into the container via the `APPTAINERENV_*` prefix.
- Binds your host cache directory to `/opt/nim/.cache` so model downloads persist across runs.
- Uses `--nv` for GPU access and `--writable-tmpfs` for a writable `/tmp` inside the container.

**Monitor the job:**

```bash
squeue -u $USER
tail -f nim-<jobid>.out
```

The endpoint is ready once you see `Uvicorn running on http://0.0.0.0:8000` in the log.

In [ ]:
%writefile 01_deply_nim.sh
#!/usr/bin/env bash
#SBATCH --job-name=llama32
#SBATCH --account=tutorial
#SBATCH --partition=tutorial
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --gres=gpu:1
#SBATCH --time=1:00:00
#SBATCH --output=nim-%j.out
#SBATCH --error=nim-%j.err

set -Eeuo pipefail

SCRIPT_DIR="/net/people/tutorial/$USER"

IMAGE="PATH_TO_IMAGE_DIRECTORY/llama-3.2-3b-instruct.sif"
CACHE_DIR="PATH_TO_CACHE_DIRECTORY/.cache"

mkdir -p "${CACHE_DIR}"
export APPTAINERENV_NGC_API_KEY="NIM_API_KEY"
export APPTAINERENV_NIM_CACHE_PATH="/opt/nim/.cache"
export APPTAINERENV_NIM_HTTP_API_PORT="8000"
export APPTAINERENV_TMPDIR="/tmp"

exec apptainer run --nv --writable-tmpfs \
    --bind "${CACHE_DIR}:/opt/nim/.cache" \
    "${IMAGE}"

In [ ]:
!sbatch 01_deply_nim.sh

### Connecting to the Local NIM Endpoint

Find the node your NIM job is running and node name from the `NODELIST` column and replace `<HOST_NAME>` in the cell below before running it.

In [ ]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://<HOST_NAME>:8000", model="meta/llama-3.2-3b-instruct", temperature=0.1, max_completion_tokens=1000, top_p=1.0)

result = llm.invoke("What is 2+2?")
print(result.content)

##### Expected Output
2 + 2 = 4

### Sending a Request via the REST API

You can also hit the NIM endpoint directly using its OpenAI-compatible REST API — no LangChain required.

Replace `<HOST_NAME>` in the cell below with your compute node name before running it.

In [ ]:
!curl -X POST \
    "http://<HOST_NAME>:8000/v1/completions" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{"model": "meta/llama-3.2-3b-instruct", "prompt": "What is 2+2?", "max_tokens": 64, "temperature": 0.1, "top_p": 1.0, "stop": ["."]}'

##### Expected Output
{"id":"cmpl-7c7d7a0b885f422992ed6569cdd9feb6","object":"text_completion","created":1784588962,"model":"meta/llama-3.2-3b-instruct","choices":[{"index":0,"text":" 2+2 is 4","logprobs":null,"finish_reason":"stop","stop_reason":".","prompt_logprobs":null}],"usage":{"prompt_tokens":8,"total_tokens":16,"completion_tokens":8}}

### Exploration Steps

- Bring your own dataset and create an MCP server around it, then re-run the entire bootcamp using local NIM endpoints instead of hosted ones.
- Evaluate the quality of `llama-3.2-3b-instruct` — how well does it perform for our use case?

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.